In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time

driver = webdriver.Chrome()
driver.maximize_window()
wait = WebDriverWait(driver, 10)

driver.get("http://localhost:5173/")
driver.execute_script("window.localStorage.clear(); window.sessionStorage.clear();")
driver.get("http://localhost:5173/login")

wait.until(EC.presence_of_element_located((By.ID, "username")))
driver.find_element(By.ID, "username").send_keys("shawon@gmail.com")
driver.find_element(By.ID, "password").send_keys("12345678")
driver.find_element(By.ID, "sign-in-btn").click()

time.sleep(3)
print("URL:", driver.current_url)
print("Page text:", driver.find_element(By.TAG_NAME, "body").text[:200])

In [ ]:
try:
    # Go to Customers (sidebar button, verified in AppShell.jsx)
    wait.until(EC.element_to_be_clickable((By.XPATH, "//aside//button[contains(., 'Customers')]"))).click()
    wait.until(EC.visibility_of_element_located((By.XPATH, "//h2[text()='Customers']")))
    wait.until(EC.presence_of_element_located((By.XPATH, "//button[contains(@class, 'cust-row')]")))

    # Use an existing customer from the list as the search term
    term = driver.find_element(By.XPATH, "(//span[contains(@class, 'cust-row-name')])[1]").text
    assert term, "Customer list is empty, nothing to search for."
    print("Search term:", term)

    # Real search field (verified in CustomerDirectory.jsx)
    search = driver.find_element(By.XPATH, "//input[@aria-label='Search customers by name, phone or email']")
    search.clear()
    search.send_keys(term)
    time.sleep(2)
    body = driver.find_element(By.TAG_NAME, "body").text
    assert term in body and "No customers found" not in body, "Matching customer not shown."
    print("Result text:", term, "found in the list.")

    # Filtering check: nonsense term must show no results
    search.clear()
    search.send_keys("zzz_no_such_customer_999")
    time.sleep(2)
    assert "No customers found" in driver.find_element(By.TAG_NAME, "body").text, \
        "Unrelated search did not filter the list."
    print("Unrelated term correctly shows: No customers found")

    print("Current URL:", driver.current_url)
    print("PASS: Customer Search")
except Exception as e:
    print("FAIL: Customer Search")
    print("Error:", e)
    driver.save_screenshot("24_customer_search_FAIL.png")

In [ ]:
driver.quit()